<a href="https://colab.research.google.com/github/MamidiPravallikaReddy/DL_LAB/blob/main/DL_week12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Next char

In [1]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

vocab = ['m', 'n', 'o', 'p', '<stop>']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

sequence = ['m', 'n', 'o']
target = ['n', 'o', 'p']

X = np.eye(len(vocab))[[char2idx[c] for c in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]

model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.fit(X, y, epochs=200, verbose=0)

test = np.eye(len(vocab))[char2idx['n']].reshape(1, 1, len(vocab))
pred = model.predict(test)

print(idx2char[np.argmax(pred)])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step
o


Next word


In [4]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

vocab = ['data', 'science', 'python', 'code']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

sequence = ['data', 'science', 'python']
target = ['science', 'python', 'code']

X = np.eye(len(vocab))[[word2idx[w] for w in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[word2idx[w] for w in target]]

model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.fit(X, y, epochs=300, verbose=0)

test = np.eye(len(vocab))[word2idx['python']].reshape(1, 1, len(vocab))
pred = model.predict(test)

print("Next word prediction:", idx2word[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
Next word prediction: code


Next sentence

In [6]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

vocab = ['we', 'are', 'learning', 'deep', 'learning']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

sequence = ['we', 'are', 'learning', 'deep']
target = ['are', 'learning', 'deep', 'learning']

X = np.eye(len(vocab))[[word2idx[w] for w in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[word2idx[w] for w in target]]

model = Sequential([
    SimpleRNN(16, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.fit(X, y, epochs=400, verbose=0)

def generate_sentence(seed, steps=6):
    result = seed.copy()

    for _ in range(steps):
        x = np.eye(len(vocab))[[word2idx[result[-1]]]].reshape(1, 1, len(vocab))
        pred = model.predict(x, verbose=0)
        next_word = idx2word[np.argmax(pred)]
        result.append(next_word)

    return " ".join(result)

print("Generated sentence:")
print(generate_sentence(['we']))

Generated sentence:


we are learning deep learning deep learning


LSTM Model

In [8]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np

vocab = {'we': 0, 'are': 1, 'learning': 2, 'deep': 3, 'learning_ai': 4}
idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

X = np.array([[0, 1, 2, 3]])
y = np.array([4])

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10, input_length=4),
    LSTM(32),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit(X, y, epochs=200, verbose=0)

test = np.array([[0, 1, 2, 3]])
pred = model.predict(test)

print("Next word prediction:", idx2word[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
Next word prediction: learning_ai


GRU

In [9]:
from keras.models import Sequential
from keras.layers import Embedding, GRU, Dense
import numpy as np

vocab = {'we': 0, 'are': 1, 'learning': 2, 'deep': 3, 'learning_ai': 4}
idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

X = np.array([[0, 1, 2, 3]])
y = np.array([4])

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10, input_length=4),
    GRU(32),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit(X, y, epochs=200, verbose=0)

test = np.array([[0, 1, 2, 3]])
pred = model.predict(test)

print("Next word prediction:", idx2word[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
Next word prediction: learning_ai


ENCODER and DECODER

In [11]:
import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Embedding, Dense

input_texts = ["i am", "you are", "he is", "she is"]
target_texts = ["je suis", "tu es", "il est", "elle est"]

input_vocab = sorted(set(" ".join(input_texts).split()))
target_vocab = sorted(set(" ".join(target_texts).split()))

input_token = {w:i for i,w in enumerate(input_vocab)}
target_token = {w:i for i,w in enumerate(target_vocab)}
inv_target_token = {i:w for w,i in target_token.items()}

encoder_input = np.array([[input_token[w] for w in s.split()] for s in input_texts])
decoder_input = np.array([[target_token[w] for w in s.split()] for s in target_texts])
decoder_target = np.array([[target_token[w] for w in s.split()] for s in target_texts])

latent_dim = 16

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(len(input_vocab), 8)(encoder_inputs)
_, h, c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [h, c]

decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(len(target_vocab), 8)(decoder_inputs)
dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_out, _, _ = dec_lstm(dec_emb, initial_state=encoder_states)
outputs = Dense(len(target_vocab), activation='softmax')(dec_out)

model = Model([encoder_inputs, decoder_inputs], outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit([encoder_input, decoder_input], decoder_target, epochs=300, verbose=0)

def translate(sentence):
    seq = np.array([[input_token[w] for w in sentence.split()]])
    pred = model.predict([seq, np.zeros((1, len(sentence.split())))])
    words = [inv_target_token[np.argmax(p)] for p in pred[0]]
    return " ".join(words)

print("Input: i am")
print("Output:", translate("i am"))

Input: i am
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
Output: je je


Attention Mechanism

In [13]:
import numpy as np
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, LSTM, Embedding, Dense, Attention, Concatenate

inp = ["i am", "you are", "he is"]
out = ["je suis", "tu es", "il est"]

iv = {w:i for i,w in enumerate(set(" ".join(inp).split()))}
ov = {w:i for i,w in enumerate(set(" ".join(out).split()))}
rov = {i:w for w,i in ov.items()}

X = np.array([[iv[w] for w in s.split()] for s in inp])
Y = np.array([[ov[w] for w in s.split()] for s in out])

enc_in = Input((None,))
e = Embedding(len(iv), 8)(enc_in)
enc_out, h, c = LSTM(16, return_sequences=True, return_state=True)(e)

dec_in = Input((None,))
d = Embedding(len(ov), 8)(dec_in)
d_out, _, _ = LSTM(16, return_sequences=True, return_state=True)(d, initial_state=[h,c])

att = Attention()([d_out, enc_out])
x = Concatenate()([d_out, att])

out_layer = Dense(len(ov), activation='softmax')(x)

model = Model([enc_in, dec_in], out_layer)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.fit([X, Y], Y, epochs=200, verbose=0)

def pred(s):
    s = np.array([[iv[w] for w in s.split()]])
    p = model.predict([s, np.zeros((1, len(s[0])))], verbose=0)
    return " ".join([rov[np.argmax(i)] for i in p[0]])

print(pred("i am"))

je suis
